## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print('Libraries loaded.')

Libraries loaded.


## 2. Load All Datasets

In [ ]:
# Download data files from Google Drive

!pip install gdown -q
import gdown

# spotify data
gdown.download('https://drive.google.com/uc?id=1_1gQKDd-2RuQXOVTRBKHcyndxwcCcPrc',
               'Spotify_data.xlsx', quiet=False, fuzzy=True)
# spotify history
gdown.download('https://drive.google.com/uc?id=1vl3YyKYAcazsQmjObePPCoCO5PYoP6J3',
               'spotify_history.csv', quiet=False)
# streaming activity
gdown.download('https://drive.google.com/uc?id=1Qh0ZoIgrRS0fSooBp7U53cHzm4RL7Vyp',
               'My_Streaming_Activity.csv', quiet=False)
# spotify recs
gdown.download('https://drive.google.com/uc?id=1LgtifR5DqKPe0PTMbvUlJFU74Rv_uPx2',
               'spotify_recommendations.csv', quiet=False)

print('All files downloaded.') # confirm

# Dataset 1: User Behavior Survey
# Source: https://www.kaggle.com/datasets/meeraajayakumar/spotify-user-behavior-dataset
# 520 survey respondents. Genre preferences, listening habits, mood associations
df_behavior = pd.read_excel('Spotify_data.xlsx')
print(f'Behavior dataset: {df_behavior.shape[0]:>6} rows, {df_behavior.shape[1]} cols')

# Dataset 2: User 1 Streaming History
# Source: https://www.kaggle.com/datasets/sgoutami/spotify-streaming-history
# ~150k plays with track URI, timestamp, platform, ms played, skip/shuffle info
# Same format as participant JSON exports we will collect
df_history = pd.read_csv('spotify_history.csv')
print(f'Streaming history: {df_history.shape[0]:>6} rows, {df_history.shape[1]} cols')

# Dataset 3: User 2 Personal Streaming Activity
# A second user's real Spotify streaming history (2017-2021)
# Contains song name, performer, album, timestamps. Used for Model 3 (user-user similarity)
df_mystream = pd.read_csv('My_Streaming_Activity.csv')
print(f'My streaming activity: {df_mystream.shape[0]:>6} rows, {df_mystream.shape[1]} cols')

# Dataset 4: Audio Features + Liked Label
# 195 songs with Spotify audio features and a binary liked label (1=liked, 0=not liked)
# Primary training data for Model 1 (KNN)
df_rec = pd.read_csv('spotify_recommendations.csv')
print(f'Audio features/liked: {df_rec.shape[0]:>6} rows, {df_rec.shape[1]} cols')

Downloading...
From: https://drive.google.com/uc?id=1_1gQKDd-2RuQXOVTRBKHcyndxwcCcPrc
To: /content/Spotify_data.xlsx
100%|██████████| 53.3k/53.3k [00:00<00:00, 23.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vl3YyKYAcazsQmjObePPCoCO5PYoP6J3
To: /content/spotify_history.csv
100%|██████████| 21.3M/21.3M [00:00<00:00, 112MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Qh0ZoIgrRS0fSooBp7U53cHzm4RL7Vyp
To: /content/My_Streaming_Activity.csv
100%|██████████| 8.04M/8.04M [00:00<00:00, 159MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LgtifR5DqKPe0PTMbvUlJFU74Rv_uPx2
To: /content/spotify_recommendations.csv
100%|██████████| 14.4k/14.4k [00:00<00:00, 29.7MB/s]


All files downloaded.
Behavior dataset:    520 rows, 20 cols
Streaming history: 149860 rows, 11 cols
My streaming activity:  62907 rows, 7 cols
Audio features/liked:    195 rows, 14 cols


## 3. Initial Inspection

In [ ]:
# print summary for all the datasets

for name, df in [('Behavior', df_behavior), ('History', df_history),
                 ('My Stream', df_mystream), ('Recommendations', df_rec)]:
    print(f'\n{name}')
    print('Columns: ', df.columns.tolist())
    print('Duplicates: ', df.duplicated().sum())
    nulls = df.isnull().sum()
    print('Nulls: ', nulls[nulls > 0].to_dict() if nulls.any() else 'None')


Behavior
Columns:  ['Age', 'Gender', 'spotify_usage_period', 'spotify_listening_device', 'spotify_subscription_plan', 'premium_sub_willingness', 'preffered_premium_plan', 'preferred_listening_content', 'fav_music_genre', 'music_time_slot', 'music_Influencial_mood', 'music_lis_frequency', 'music_expl_method', 'music_recc_rating', 'pod_lis_frequency', 'fav_pod_genre', 'preffered_pod_format', 'pod_host_preference', 'preffered_pod_duration', 'pod_variety_satisfaction']
Duplicates:  1
Nulls:  {'preffered_premium_plan': 208, 'fav_pod_genre': 148, 'preffered_pod_format': 140, 'pod_host_preference': 141, 'preffered_pod_duration': 129}

History
Columns:  ['spotify_track_uri', 'ts', 'platform', 'ms_played', 'track_name', 'artist_name', 'album_name', 'reason_start', 'reason_end', 'shuffle', 'skipped']
Duplicates:  1185
Nulls:  {'reason_start': 143, 'reason_end': 117}

My Stream
Columns:  ['index', 'SongID', 'TimeStamp_Central', 'Performer', 'Album', 'Song', 'TimeStamp_UTC']
Duplicates:  0
Nulls:

## 4. Clean: User Behavior Dataset (`Spotify_data.xlsx`)

**Issues found:**
- 1 duplicate row
- Misspelled column names (`preffered`, `Influencial`)
- `'None'` strings masking real null values
- Multi-value entries in some columns

In [ ]:
df_beh = df_behavior.copy()

# remove duplicate row
before = len(df_beh)
df_beh = df_beh.drop_duplicates()
print(f'Removed {before - len(df_beh)} duplicate row(s). Rows remaining: {len(df_beh)}')

Removed 1 duplicate row(s). Rows remaining: 519


In [ ]:
# fix misspelled and inconsistent column names
df_beh = df_beh.rename(columns={
    'preffered_premium_plan' : 'preferred_premium_plan',
    'preffered_pod_format'   : 'preferred_pod_format',
    'preffered_pod_duration' : 'preferred_pod_duration',
    'music_Influencial_mood' : 'music_influential_mood',
    'music_lis_frequency'    : 'music_listen_frequency',
    'music_expl_method'      : 'music_explore_method',
    'music_recc_rating'      : 'music_recommendation_rating',
    'pod_lis_frequency'      : 'pod_listen_frequency',
    'fav_pod_genre'          : 'fav_podcast_genre',
})
print('Columns after rename:')
print(df_beh.columns.tolist())

Columns after rename:
['Age', 'Gender', 'spotify_usage_period', 'spotify_listening_device', 'spotify_subscription_plan', 'premium_sub_willingness', 'preferred_premium_plan', 'preferred_listening_content', 'fav_music_genre', 'music_time_slot', 'music_influential_mood', 'music_listen_frequency', 'music_explore_method', 'music_recommendation_rating', 'pod_listen_frequency', 'fav_podcast_genre', 'preferred_pod_format', 'pod_host_preference', 'preferred_pod_duration', 'pod_variety_satisfaction']


In [ ]:
# strip whitespace and replace 'None' strings with NaN
str_cols = df_beh.select_dtypes(include='object').columns
df_beh[str_cols] = df_beh[str_cols].apply(lambda col: col.str.strip())
df_beh.replace('None', np.nan, inplace=True)

nulls = df_beh.isnull().sum()
print('Nulls after None -> NaN replacement:')
print(nulls[nulls > 0] if nulls.any() else 'No nulls found.')

Nulls after None -> NaN replacement:
preferred_premium_plan    207
fav_podcast_genre         147
preferred_pod_format      139
pod_host_preference       140
preferred_pod_duration    128
dtype: int64


In [ ]:
# check key column distributions
key_cols = ['fav_music_genre', 'music_time_slot',
            'music_listen_frequency', 'music_influential_mood']
for col in key_cols:
    print(f'\n{col}')
    print(df_beh[col].value_counts())


fav_music_genre
fav_music_genre
Melody                       258
classical                     87
Pop                           85
Rap                           55
Electronic/Dance              16
All                            6
Rock                           4
Kpop                           4
Classical & melody, dance      2
Old songs                      1
trending songs random          1
Name: count, dtype: int64

music_time_slot
music_time_slot
Night        311
Afternoon    117
Morning       91
Name: count, dtype: int64

music_listen_frequency
music_listen_frequency
While Traveling                                                               111
leisure time                                                                   87
While Traveling, leisure time                                                  64
While Traveling, Workout session, leisure time                                 48
Workout session                                                                33
Study Hours

In [ ]:
print('Cleaned behavior dataset shape:', df_beh.shape)
df_beh.head(3)

Cleaned behavior dataset shape: (519, 20)


,Age,Gender,spotify_usage_period,spotify_listening_device,spotify_subscription_plan,premium_sub_willingness,preferred_premium_plan,preferred_listening_content,fav_music_genre,music_time_slot,music_influential_mood,music_listen_frequency,music_explore_method,music_recommendation_rating,pod_listen_frequency,fav_podcast_genre,preferred_pod_format,pod_host_preference,preferred_pod_duration,pod_variety_satisfaction
0,20-35,Female,More than 2 years,Smart speakers or voice assistants,Free (ad-supported),Yes,Family Plan-Rs 179/month,Podcast,Melody,Night,Sadness or melancholy,leisure time,Playlists,3,Daily,Comedy,Interview,Both,Both,Ok
1,12-20,Male,More than 2 years,Computer or laptop,Free (ad-supported),Yes,Individual Plan- Rs 119/ month,Podcast,Rap,Afternoon,Social gatherings or parties,Workout session,Playlists,2,Several times a week,Comedy,Interview,Both,NaN,Satisfied
2,35-60,Others,6 months to 1 year,Smart speakers or voice assistants,Free (ad-supported),Yes,Student Plan-Rs 59/month,Podcast,Pop,Night,Relaxation and stress relief,"Study Hours, While Traveling",Playlists,4,Once a week,Sports,Interview,NaN,Both,Satisfied


## 5. Clean: User 1 Streaming History (spotify_history.csv)

**Issues found:**
- 1,185 duplicate rows
- `ts` stored as string: needs datetime parsing
- `ms_played` in milliseconds: needs conversion
- Rows where `ms_played == 0` (accidental/buffering plays)
- 143 nulls in `reason_start`, 117 in `reason_end`
- Plays under 30s are implicit skips even when `skipped == False`

In [ ]:
df_hist = df_history.copy()

# remove duplicates
before = len(df_hist)
df_hist = df_hist.drop_duplicates()
print(f'Removed {before - len(df_hist)} duplicate rows. Rows remaining: {len(df_hist)}')

Removed 1185 duplicate rows. Rows remaining: 148675


In [ ]:
# parse timestamp
df_hist['ts'] = pd.to_datetime(df_hist['ts'])
print('ts dtype:', df_hist['ts'].dtype)
print('Date range:', df_hist['ts'].min(), 'to', df_hist['ts'].max())

ts dtype: datetime64[ns]
Date range: 2013-07-08 02:44:34 to 2024-12-15 23:06:25


In [ ]:
# convert ms_played to seconds and minutes
df_hist['seconds_played'] = df_hist['ms_played'] / 1000
df_hist['minutes_played'] = df_hist['seconds_played'] / 60
print('minutes_played stats:')
print(df_hist['minutes_played'].describe())

minutes_played stats:
count    148675.000000
mean          2.134345
std           1.963629
min           0.000000
25%           0.046433
50%           2.306433
75%           3.638883
max          26.018750
Name: minutes_played, dtype: float64


In [ ]:
# remove zero-play rows
zero_count = (df_hist['ms_played'] == 0).sum()
print(f'Zero-play rows: {zero_count}')
df_hist = df_hist[df_hist['ms_played'] > 0]
print(f'Shape after removal: {df_hist.shape}')

Zero-play rows: 3536
Shape after removal: (145139, 13)


In [ ]:
# flag likely skips (plays under 30 seconds)
df_hist['likely_skipped'] = df_hist['seconds_played'] < 30
print(f'Plays under 30s: {df_hist["likely_skipped"].sum()} ({df_hist["likely_skipped"].mean()*100:.1f}%)')

Plays under 30s: 51815 (35.7%)


In [ ]:
# fill nulls in reason_start and reason_end
df_hist['reason_start'] = df_hist['reason_start'].fillna('unknown')
df_hist['reason_end']   = df_hist['reason_end'].fillna('unknown')
print('Nulls remaining:', df_hist.isnull().sum().sum())

Nulls remaining: 0


In [ ]:
# extract time features from timestamp
df_hist['hour']        = df_hist['ts'].dt.hour
df_hist['day_of_week'] = df_hist['ts'].dt.day_name()
df_hist['month']       = df_hist['ts'].dt.month
df_hist['year']        = df_hist['ts'].dt.year
print('Time features added: hour, day_of_week, month, year')

Time features added: hour, day_of_week, month, year


In [ ]:
# derive skip_rate and play_count per track
skip_rate  = df_hist.groupby('track_name')['likely_skipped'].mean().reset_index()
skip_rate.columns = ['track_name', 'skip_rate']
play_count = df_hist.groupby('track_name').size().reset_index(name='play_count')

df_hist = df_hist.merge(skip_rate,  on='track_name', how='left')
df_hist = df_hist.merge(play_count, on='track_name', how='left')

print('Top 5 most played tracks:')
print(df_hist[['track_name', 'artist_name', 'play_count']]
      .drop_duplicates().sort_values('play_count', ascending=False).head(5))

Top 5 most played tracks:
                               track_name     artist_name  play_count
66847                     Ode To The Mets     The Strokes         201
14599                        In the Blood      John Mayer         179
72209                         Dying Breed     The Killers         163
65464                             Caution     The Killers         153
112332  19 Dias y 500 Noches - En Directo  Joaquín Sabina         145


In [ ]:
print('Cleaned streaming history shape:', df_hist.shape)
df_hist.head(3)

Cleaned streaming history shape: (145139, 20)


,spotify_track_uri,ts,platform,ms_played,track_name,artist_name,album_name,reason_start,reason_end,shuffle,skipped,seconds_played,minutes_played,likely_skipped,hour,day_of_week,month,year,skip_rate,play_count
0,2J3n32GeLmMjwuAzyhcSNe,2013-07-08 02:44:34,web player,3185,"Say It, Just Say It",The Mowgli's,Waiting For The Dawn,autoplay,clickrow,False,False,3.185,0.053083,True,2,Monday,7,2013,1.0,1
1,1oHxIPqJyvAYHy0PVrDU98,2013-07-08 02:45:37,web player,61865,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris,18 Months,clickrow,clickrow,False,False,61.865,1.031083,False,2,Monday,7,2013,0.0,2
2,487OPlneJNni3NWC8SYqhW,2013-07-08 02:50:24,web player,285386,Born To Die,Lana Del Rey,Born To Die - The Paradise Edition,clickrow,unknown,False,False,285.386,4.756433,False,2,Monday,7,2013,0.4,5


## 6. Clean: User 2 Streaming Activity (My_Streaming_Activity.csv)

**Issues found:**
- No duplicates
- 2,559 nulls in `Album`: these are non-music entries (videos, gaming clips, podcasts)
- `SongID` is a concatenation of Song + Artist with no separator: not parseable
- Two timestamp columns: only UTC needed
- Column names differ from `df_hist`: needs renaming for Model 3 consistency

In [ ]:
df_myst = df_mystream.copy()

# drop redundant columns
df_myst = df_myst.drop(columns=['index', 'TimeStamp_Central', 'SongID'])
print('Columns after drop:', df_myst.columns.tolist())

Columns after drop: ['Performer', 'Album', 'Song', 'TimeStamp_UTC']


In [ ]:
# parse timestamp and rename to match df_hist
df_myst['TimeStamp_UTC'] = pd.to_datetime(df_myst['TimeStamp_UTC'])
df_myst = df_myst.rename(columns={'TimeStamp_UTC': 'ts'})
print('ts dtype:', df_myst['ts'].dtype)
print('Date range:', df_myst['ts'].min(), 'to', df_myst['ts'].max())

/tmp/ipykernel_9388/2359110806.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_myst['TimeStamp_UTC'] = pd.to_datetime(df_myst['TimeStamp_UTC'])


ts dtype: datetime64[ns]
Date range: 2017-01-01 15:51:00 to 2021-05-25 23:18:00


In [ ]:
# flag non-music rows (null Album)
df_myst['is_non_music'] = df_myst['Album'].isnull()
print(f'Non-music rows flagged: {df_myst["is_non_music"].sum()}')
print('Sample non-music entries:')
print(df_myst[df_myst['is_non_music']][['Song', 'Performer']].head(5))

Non-music rows flagged: 2559
Sample non-music entries:
                                           Song  \
3410           Bloopers/Gag Reel Part 2 (1080p)   
3411   Between Two Ferns with Zach Galifianakis   
3412   Between Two Ferns with Zach Galifianakis   
4546                                         Up   
4835  Shae Katha Shrine + Farosh Scale Hunting!   

                                Performer  
3410                          Anchorman 2  
3411         Conan O'Brien & Andy Richter  
3412                            Awkwafina  
4546  Office Chair vs. Gaming Chair Round  
4835             Zelda Breath of the Wild  


In [ ]:
# strip whitespace from strings
str_cols = df_myst.select_dtypes(include='object').columns
df_myst[str_cols] = df_myst[str_cols].apply(lambda col: col.str.strip())
print('Whitespace stripped.')

Whitespace stripped.


In [ ]:
# extract time features
df_myst['hour']        = df_myst['ts'].dt.hour
df_myst['day_of_week'] = df_myst['ts'].dt.day_name()
df_myst['month']       = df_myst['ts'].dt.month
df_myst['year']        = df_myst['ts'].dt.year
print('Time features added.')

Time features added.


In [ ]:
# derive play_count per track
play_count2 = df_myst.groupby('Song').size().reset_index(name='play_count')
df_myst = df_myst.merge(play_count2, on='Song', how='left')
print('Top 5 most played songs:')
print(df_myst[['Song', 'Performer', 'play_count']]
      .drop_duplicates().sort_values('play_count', ascending=False).head(5))

Top 5 most played songs:
           Song        Performer  play_count
927     Thunder  Imagine Dragons         169
933    Believer  Imagine Dragons         161
14886  Believer   Above & Beyond         161
629      Issues   Julia Michaels         145
12078    Issues   Violent Femmes         145


In [ ]:
# rename columns to match df_hist convention
df_myst = df_myst.rename(columns={
    'Song'      : 'track_name',
    'Performer' : 'artist_name',
    'Album'     : 'album_name'
})
print('Cleaned User 2 streaming activity shape:', df_myst.shape)
df_myst.head(3)

Cleaned User 2 streaming activity shape: (62907, 10)


,artist_name,album_name,track_name,ts,is_non_music,hour,day_of_week,month,year,play_count
0,Edwin Starr,25 Miles,Twenty Five Miles,2021-05-25 23:18:00,False,23,Tuesday,5,2021,2
1,Greyhounds,Change of Pace,Devil's Eyes,2021-05-25 23:15:00,False,23,Tuesday,5,2021,3
2,Murs,Have a Nice Life,Pussy and Pizza,2021-05-25 23:12:00,False,23,Tuesday,5,2021,3


## 7. Clean: Audio Features + Liked Label (spotify_recommendations.csv)

**Issues found:**
- No duplicates, no nulls: already clean
- Audio features on different scales: needs normalization before KNN
- `liked` is binary (1/0) and nearly balanced (100 liked, 95 not liked)
- `duration_ms` should be converted to seconds for consistency

In [ ]:
df_audio = df_rec.copy()

# convert duration_ms to seconds
df_audio['duration_s'] = df_audio['duration_ms'] / 1000
df_audio = df_audio.drop(columns=['duration_ms'])
print('duration_ms converted to duration_s.')

print('\nClass balance (liked):')
print(df_audio['liked'].value_counts())
print(f'Liked: {df_audio["liked"].mean()*100:.1f}% | Not liked: {(1-df_audio["liked"].mean())*100:.1f}%')

duration_ms converted to duration_s.

Class balance (liked):
liked
1    100
0     95
Name: count, dtype: int64
Liked: 51.3% | Not liked: 48.7%


In [ ]:
# normalize audio features using MinMax scaling

feature_cols = ['danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'duration_s', 'time_signature']

scaler = MinMaxScaler()
df_audio[feature_cols] = scaler.fit_transform(df_audio[feature_cols])

print('Features normalized to [0, 1]. Verify ranges:')
print(df_audio[feature_cols].describe().loc[['min', 'max']])

Features normalized to [0, 1]. Verify ranges:
     danceability  energy  key  loudness  mode  speechiness  acousticness  \
min           0.0     0.0  0.0       0.0   0.0          0.0           0.0   
max           1.0     1.0  1.0       1.0   1.0          1.0           1.0   

     instrumentalness  liveness  valence  tempo  duration_s  time_signature  
min               0.0       0.0      0.0    0.0         0.0             0.0  
max               1.0       1.0      1.0    1.0         1.0             1.0  


In [ ]:
print('Cleaned audio features shape:', df_audio.shape)
df_audio.head(3)

Cleaned audio features shape: (195, 14)


,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,liked,duration_s
0,0.824755,0.625604,0.636364,0.889092,0.0,0.038852,0.453265,0.000757,0.111519,0.627395,0.298644,0.75,0,0.393282
1,0.774510,0.705113,0.909091,0.859361,0.0,0.543147,0.207033,0.000000,0.096849,0.512014,0.760506,0.75,1,0.294069
2,0.160539,0.012581,0.090909,0.369017,1.0,0.027528,0.996985,0.925697,0.114852,0.003070,0.126184,0.75,0,0.362942


In [ ]:
# combine both streaming histories into one

# Add a user identifier so we can tell them apart later
df_hist['user_id'] = 'user_1'
df_myst['user_id'] = 'user_2'

# Keep only columns that exist in both
shared_cols = ['track_name', 'artist_name', 'ts', 'hour',
               'day_of_week', 'month', 'year', 'play_count', 'user_id']

df_combined = pd.concat([df_hist[shared_cols], df_myst[shared_cols]], ignore_index=True)

print(f'Combined streaming dataset: {df_combined.shape[0]} rows, {df_combined.shape[1]} cols')
print(f'Users: {df_combined["user_id"].value_counts().to_dict()}')

Combined streaming dataset: 208046 rows, 9 cols
Users: {'user_1': 145139, 'user_2': 62907}


## 8. Save All Cleaned Datasets

Save to your shared Google Drive folder. All other notebooks will load from here.

In [ ]:
# Save cleaned files to Colab's temporary storage
OUTPUT_PATH = '/content/'

df_beh.to_csv(f'{OUTPUT_PATH}cleaned_user_behavior.csv',          index=False)
df_hist.to_csv(f'{OUTPUT_PATH}cleaned_streaming_history.csv',     index=False)
df_myst.to_csv(f'{OUTPUT_PATH}cleaned_my_streaming_activity.csv', index=False)
df_audio.to_csv(f'{OUTPUT_PATH}cleaned_audio_features.csv',       index=False)

print('Saved:')
print(f'  cleaned_user_behavior.csv          ({len(df_beh)} rows)')
print(f'  cleaned_streaming_history.csv      ({len(df_hist)} rows)')
print(f'  cleaned_my_streaming_activity.csv  ({len(df_myst)} rows)')
print(f'  cleaned_audio_features.csv         ({len(df_audio)} rows)')

# Download all files to computer
from google.colab import files
files.download('/content/cleaned_user_behavior.csv')
files.download('/content/cleaned_streaming_history.csv')
files.download('/content/cleaned_my_streaming_activity.csv')
files.download('/content/cleaned_audio_features.csv')

Saved:
  cleaned_user_behavior.csv          (519 rows)
  cleaned_streaming_history.csv      (145139 rows)
  cleaned_my_streaming_activity.csv  (62907 rows)
  cleaned_audio_features.csv         (195 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>